# CodeLens — Embed codebase into Qdrant

This notebook:
1. Reads all 20 functions from Neo4j
2. Builds rich text chunks for each
3. Embeds them with OpenAI
4. Stores in Qdrant
5. Tests a natural language search

**Run after** `codelens_ingest.ipynb` — Neo4j must already have data.

## Cell 1 — Install dependencies

In [1]:
%pip install qdrant-client openai neo4j --quiet
print("Done")

Note: you may need to restart the kernel to use updated packages.
Done



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Config

In [1]:
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from openai import OpenAI
import hashlib
import os
from dotenv import load_dotenv

load_dotenv() 

# ── Config ───────────────────────────────────────────────
NEO4J_URI      = os.getenv("NEO4J_URI")  # e.g. "bolt://localhost:7687"
NEO4J_USER     = os.getenv("NEO4J_USER") # e.g. "neo4j"
NEO4J_PASS     = os.getenv("NEO4J_PASS")    # ← your Neo4j password
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")         # ← your OpenAI key
QDRANT_URL     = "http://localhost:6333"
COLLECTION     = "test_codebase"  # Qdrant collection name
EMBED_MODEL    = "text-embedding-3-small"  # 1536 dims, cheapest
# ─────────────────────────────────────────────────────────

# Test all three connections
try:
    neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    neo4j_driver.verify_connectivity()
    print("Neo4j     ✓")
except Exception as e:
    print(f"Neo4j     ✗  {e}")

try:
    qdrant = QdrantClient(url=QDRANT_URL)
    qdrant.get_collections()
    print("Qdrant    ✓")
except Exception as e:
    print(f"Qdrant    ✗  {e}")

try:
    openai = OpenAI(api_key=OPENAI_API_KEY)
    openai.models.list()
    print("OpenAI    ✓")
except Exception as e:
    print(f"OpenAI    ✗  {e}")

Neo4j     ✗  Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Qdrant    ✗  [WinError 10061] No connection could be made because the target machine actively refused it


c:\Users\Mohammad\AppData\Local\Programs\Python\Python312\Lib\site-packages\qdrant_client\qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


OpenAI    ✓


## Cell 3 — Pull all functions from Neo4j

In [3]:
def fetch_all_functions() -> list[dict]:
    """
    Pull every function node from Neo4j with all its properties.
    Also fetches what each function CALLS so we can include that context.
    """
    cypher = """
    MATCH (f:Function)
    OPTIONAL MATCH (f)-[:CALLS]->(callee:Function)
    OPTIONAL MATCH (f)-[:CALLS_EXTERNAL]->(ext:ExternalLib)
    OPTIONAL MATCH (f)-[:BELONGS_TO]->(c:Class)
    RETURN
        f.id          AS id,
        f.name        AS name,
        f.file        AS file,
        f.start_line  AS start_line,
        f.end_line    AS end_line,
        f.class_name  AS class_name,
        f.docstring   AS docstring,
        f.params      AS params,
        f.return_type AS return_type,
        f.is_async    AS is_async,
        f.source      AS source,
        collect(DISTINCT callee.name) AS calls_internal,
        collect(DISTINCT ext.name)    AS calls_external
    """
    with neo4j_driver.session() as s:
        results = [dict(r) for r in s.run(cypher)]
    return results


functions = fetch_all_functions()

print(f"Fetched {len(functions)} functions from Neo4j\n")
for fn in functions:
    calls_in  = [c for c in fn['calls_internal'] if c]
    calls_ext = [c for c in fn['calls_external'] if c]
    cls       = f"[{fn['class_name']}]" if fn['class_name'] else ""
    print(f"  {fn['file']}:{fn['start_line']:<5} {fn['name']} {cls}")
    if calls_in:
        print(f"    → calls: {calls_in}")
    if calls_ext:
        print(f"    → extern: {calls_ext[:4]}")

Fetched 20 functions from Neo4j

  config.py:19    load_config 
    → extern: ['Config']
  config.py:13    get_settings [Config]
  main.py:8     main 
    → calls: ['load_config', 'is_admin', 'place_order', 'create_user', 'calculate_total']
    → extern: ['print', 'UserService', 'OrderService']
  models/order.py:8     __init__ [Order]
  models/order.py:15    process_payment [Order]
    → calls: ['calculate_total']
    → extern: ['print']
  models/order.py:24    cancel [Order]
    → extern: ['print']
  models/user.py:8     __init__ [BaseModel]
    → extern: ['datetime.now']
  models/user.py:12    save [BaseModel]
    → extern: ['print']
  models/user.py:19    __init__ [User]
    → calls: ['__init__']
    → extern: ['super']
  models/user.py:26    is_admin [User]
  models/user.py:30    add_order [User]
    → extern: ['print', 'self.orders.append']
  services/order_service.py:11    __init__ [OrderService]
    → extern: ['UserService']
  services/order_service.py:14    place_order [OrderSe

## Cell 4 — Build rich text chunks

This is the most important step. What we embed determines what questions we can answer.
We pack in every piece of context: file, class, params, docstring, source, what it calls.

In [4]:
def build_chunk(fn: dict) -> str:
    """
    Build a rich text representation of a function for embedding.
    The richer this is, the better the semantic search will be.
    """
    lines = []

    # ── Header ────────────────────────────────────────────
    lines.append(f"Function: {fn['name']}")
    lines.append(f"File: {fn['file']} (lines {fn['start_line']}-{fn['end_line']})")

    if fn.get('class_name'):
        lines.append(f"Class: {fn['class_name']}")

    if fn.get('is_async'):
        lines.append("Type: async function")

    if fn.get('params'):
        lines.append(f"Parameters: {fn['params']}")

    if fn.get('return_type'):
        lines.append(f"Returns: {fn['return_type']}")

    # ── Docstring ─────────────────────────────────────────
    if fn.get('docstring'):
        lines.append(f"Description: {fn['docstring']}")

    # ── Relationships (graph context) ─────────────────────
    calls_in = [c for c in fn.get('calls_internal', []) if c]
    if calls_in:
        lines.append(f"Calls internally: {', '.join(calls_in)}")

    calls_ext = [c for c in fn.get('calls_external', []) if c]
    if calls_ext:
        # Strip self.x noise, keep meaningful ones
        meaningful = [c for c in calls_ext if '.' not in c or not c.startswith('self')]
        if meaningful:
            lines.append(f"Uses: {', '.join(meaningful[:6])}")

    # ── Source code ───────────────────────────────────────
    if fn.get('source'):
        lines.append(f"\nSource code:\n{fn['source']}")

    return "\n".join(lines)


# Build chunks for all 20 functions
chunks = [(fn, build_chunk(fn)) for fn in functions]

# Print a sample so you can see what gets embedded
print("═" * 60)
print("SAMPLE CHUNK (this is what gets embedded)")
print("═" * 60)
# Show the most interesting one — a service method
sample = next((c for fn, c in chunks if fn['name'] == 'place_order'), chunks[0][1])
print(sample)
print()
print(f"Total chunks to embed: {len(chunks)}")

════════════════════════════════════════════════════════════
SAMPLE CHUNK (this is what gets embedded)
════════════════════════════════════════════════════════════
Function: place_order
File: services/order_service.py (lines 14-31)
Class: OrderService
Parameters: self, username, items, total
Returns: Order
Description: Place a new order for a user.
Calls internally: process_payment, add_order, create_user, get_user, format_currency
Uses: print, len, Order

Source code:
def place_order(self, username: str, items: list, total: float) -> Order:
        """Place a new order for a user."""
        user = self.user_service.get_user(username)
        if not user:
            user = self.user_service.create_user(username, f"{username}@example.com")
        
        order = Order(order_id=f"ORD-{len(user.orders)+1:04d}", 
                     user=user, 
                     items=items, 
                     total=total)
        
        # Cross-module and cross-file call
        success = ord

## Cell 5 — Create Qdrant collection

In [6]:
# Delete collection if it exists (clean re-run)
existing = [c.name for c in qdrant.get_collections().collections]
if COLLECTION in existing:
    qdrant.delete_collection(COLLECTION)
    print(f"Deleted existing collection: {COLLECTION}")

# Create fresh collection
# text-embedding-3-small produces 1536-dimensional vectors
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=1536,
        distance=Distance.COSINE
    )
)

print(f"Created collection: {COLLECTION}")
print(f"Vector size: 1536 (text-embedding-3-small)")
print(f"Distance: COSINE")

Created collection: test_codebase
Vector size: 1536 (text-embedding-3-small)
Distance: COSINE


## Cell 6 — Embed all chunks with OpenAI

In [7]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    """
    Embed a batch of texts using OpenAI.
    text-embedding-3-small: $0.02 per 1M tokens — 20 functions costs fractions of a cent.
    """
    response = openai.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]


def make_point_id(node_id: str) -> int:
    """
    Qdrant needs integer IDs. Hash the string node_id to a stable int.
    """
    return int(hashlib.md5(node_id.encode()).hexdigest(), 16) % (2**63)


# Extract just the text strings for embedding
texts      = [chunk for _, chunk in chunks]
fn_nodes   = [fn   for fn, _ in chunks]

print(f"Embedding {len(texts)} chunks...")
print(f"Model: {EMBED_MODEL}")
print(f"Estimated cost: ~$0.00 (tiny codebase)\n")

# Single batch call — all 20 functions in one API request
embeddings = embed_texts(texts)

print(f"Got {len(embeddings)} embeddings")
print(f"Each vector has {len(embeddings[0])} dimensions")

Embedding 20 chunks...
Model: text-embedding-3-small
Estimated cost: ~$0.00 (tiny codebase)



AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

## Cell 7 — Upload to Qdrant

In [ ]:
# Build Qdrant points
# Payload = metadata stored alongside the vector (no size limit)
points = [
    PointStruct(
        id=make_point_id(fn['id']),
        vector=embedding,
        payload={
            # Identity
            "node_id":     fn['id'],
            "name":        fn['name'],
            "file":        fn['file'],
            "class_name":  fn['class_name'],
            "start_line":  fn['start_line'],
            "end_line":    fn['end_line'],
            # Content
            "docstring":   fn['docstring'],
            "params":      fn['params'],
            "return_type": fn['return_type'],
            "is_async":    fn['is_async'],
            "source":      fn['source'],
            # Graph context baked in
            "calls_internal": [c for c in fn.get('calls_internal', []) if c],
            "calls_external": [c for c in fn.get('calls_external', []) if c],
            # The full rich chunk that was embedded
            "chunk_text":  texts[i],
        }
    )
    for i, (fn, embedding) in enumerate(zip(fn_nodes, embeddings))
]

# Upload all at once
qdrant.upsert(collection_name=COLLECTION, points=points)

# Verify
info = qdrant.get_collection(COLLECTION)
stored_count = (
    getattr(info, "vectors_count", None)
    or getattr(info, "points_count", None)
    or getattr(info, "indexed_vectors_count", None)
)

print(f"Uploaded {len(points)} points to Qdrant")
print(f"Collection: {COLLECTION}")
print(f"Stored items: {stored_count}")
print(f"Status: {info.status}")

## Cell 8 — Test: raw vector search
Before building the full QA system, verify search works correctly.

In [ ]:
def vector_search(question: str, top_k: int = 5) -> list[dict]:
    """
    Embed the question and find the most semantically similar functions.
    Returns ranked list of matching functions with scores.
    """
    # Embed the question using the same model
    q_embedding = openai.embeddings.create(
        model=EMBED_MODEL,
        input=[question]
    ).data[0].embedding

    # Search Qdrant
    results = qdrant.search(
        collection_name=COLLECTION,
        query_vector=q_embedding,
        limit=top_k,
        with_payload=True   # return all metadata
    )

    return results


def print_search_results(question: str, results):
    print(f"Question: '{question}'")
    print("─" * 55)
    for i, r in enumerate(results, 1):
        p    = r.payload
        cls  = f" [{p['class_name']}]" if p.get('class_name') else ""
        doc  = f"\n     doc: {p['docstring'][:70]}" if p.get('docstring') else ""
        print(f"  {i}. {p['file']}:{p['start_line']}  {p['name']}{cls}")
        print(f"     score: {r.score:.4f}{doc}")
    print()


# ── Run 5 test questions ──────────────────────────────────
test_questions = [
    "how is user creation handled?",
    "where is payment processing done?",
    "how does order placement work?",
    "where is email validation?",
    "how is admin role assigned?",
]

for question in test_questions:
    results = vector_search(question, top_k=3)
    print_search_results(question, results)
    print()

## Cell 9 — Full QA: ask anything about your codebase
Combines vector search (finds relevant functions) + OpenAI (generates answer with citations).

In [ ]:
def ask(question: str, top_k: int = 5) -> str:
    """
    Ask any natural language question about the codebase.
    Returns a grounded answer with exact file + line citations.
    """

    # ── Step 1: Find relevant functions via vector search ──
    results = vector_search(question, top_k=top_k)

    if not results:
        return "No relevant code found for that question."

    # ── Step 2: Build context block for the LLM ───────────
    context_parts = []
    for i, r in enumerate(results, 1):
        p = r.payload
        header = (
            f"[{i}] {p['file']} lines {p['start_line']}-{p['end_line']} "
            f"| function: {p['name']}"
            + (f" | class: {p['class_name']}" if p.get('class_name') else "")
            + (f" | score: {r.score:.3f}")
        )
        source = p.get('source') or "(source not available)"
        calls  = p.get('calls_internal', [])
        calls_str = f"\nCalls: {', '.join(calls)}" if calls else ""
        context_parts.append(f"{header}\n{source}{calls_str}")

    context = "\n\n".join(context_parts)

    # ── Step 3: Ask the LLM with strict citation prompt ────
    system_prompt = """You are a senior software engineer analyzing a Python codebase.
You answer developer questions using ONLY the provided code context.

Rules:
- Every claim must be backed by a specific file and line number from the context
- Use the format: `filename:line_number` for citations
- If the context doesn't contain enough info to answer, say so clearly
- Be concise and technical
- Do NOT make up code or relationships not shown in the context"""

    user_prompt = f"""Question: {question}

Code context (ranked by relevance):

{context}

Answer the question using only the code above. Cite exact file:line for every claim."""

    response = openai.chat.completions.create(
        model="gpt-4o-mini",   # cheap + smart enough for code QA
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0,         # deterministic answers
    )

    answer = response.choices[0].message.content

    # ── Step 4: Show sources used ─────────────────────────
    sources = [
        f"  [{i}] {r.payload['file']}:{r.payload['start_line']}  "
        f"{r.payload['name']}()  (score: {r.score:.3f})"
        for i, r in enumerate(results, 1)
    ]

    return (
        f"{answer}\n\n"
        f"── Sources used ──────────────────────────────────────\n"
        + "\n".join(sources)
    )


print("ask() is ready. Run Cell 10 to start asking questions.")

## Cell 10 — Ask questions about your codebase
Change the question and re-run this cell as many times as you want.

In [ ]:
# ── Change this question and re-run the cell ─────────────
question = "how is user creation handled?"
# ─────────────────────────────────────────────────────────

print(f"Q: {question}")
print("═" * 60)
answer = ask(question)
print(answer)

## Cell 11 — Try these questions on your codebase

Copy any into Cell 10's `question = ` and re-run:

```
"how is user creation handled?"
"where is payment processing done?"
"how does order placement work?"
"where is email validation?"
"how is admin role assigned to a user?"
"what does the OrderService do?"
"how is configuration loaded?"
"what happens when an order is cancelled?"
"how is the total price calculated?"
"what does the BaseModel save method do?"
"which functions handle the user model?"
"where is the entry point of the application?"
```

## Cell 12 — Ask multiple questions in a batch

In [ ]:
questions = [
    "how is user creation handled?",
    "where is payment processing done?",
    "how is admin role assigned?",
]

for q in questions:
    print(f"\nQ: {q}")
    print("═" * 60)
    print(ask(q))
    print()